# MetaPhlAn3 tutorial
MetaPhlAn relies on unique clade-specific marker genes that were identified from ~17,000 reference genomes (further details)

*   ~13,500 bacteria and archaea
*   ~3,500 viruses
*   ~110 eukaryotes

Illustration of clades [here](https://drive.google.com/file/d/1Ln_-JTfAG6fNuSl9RpLKjUQeynL229OK/view?usp=drive_link)

# Workflow:

**Input**:


*   QC'ed reads
*   Clade-specific marker gene database


**Output**:


*   Marker genes mapping file [.sam]
*   Microbial profile table [.txt]





# Setup MetaPhlAn

In [ ]:
# check if the installation is successful
import os,sys
! which bowtie2
! which metaphlan

/opt/conda/bin/bowtie2
/opt/conda/bin/metaphlan


# Input data:

In [ ]:
# mgx reads folder
root_dir = "/biodata/resources/day1_lab2"
mgx_reads_dir = os.path.join(root_dir,"mgx_reads")
! ls -lh {mgx_reads_dir}
sample_id="PSMB4MBK"
metaphlan_profile_dir =  os.path.join(root_dir,"reference_based_metagenomic_analysis","profile")
! rm -rf stdin_map.bowtie2out.txt
! mkdir -p {metaphlan_profile_dir}
! gunzip -c {mgx_reads_dir}/{sample_id}_R1.fastq.gz | head

total 780M
-rw-rw-rw- 1 root root  15K Oct  2 03:40 PSMB4MBK.log
-rw-rw-rw- 1 root root 389M Sep 30 06:54 PSMB4MBK_R1.fastq.gz
-rw-rw-rw- 1 root root 391M Sep 30 06:54 PSMB4MBK_R2.fastq.gz
-rw-rw-rw- 1 root root 482K Oct  2 03:40 PSMB4MBK_subset_R1.fastq.gz
-rw-rw-rw- 1 root root 478K Oct  2 03:40 PSMB4MBK_subset_R2.fastq.gz


## Count the number of lines and the number of reads in the fastq file

In [8]:
! zcat {mgx_reads_dir}/{sample_id}_R1.fastq.gz | wc -l

34497952


In [9]:
34497952/4

8624488.0

## Reminder: fastq format

Four lines per record [per read]


1.   Read identifier [starts with @, ends with /1 or /2]
2.   Nucleotide sequence
3.   Place holder / separater [+]
4.   Phred score [quality score for each nucleotide position]

Phred score table [here](https://drive.google.com/file/d/1poJ6joQKbhJO9btjvV5xEvRr2_0uGAxs/view?usp=drive_link)

## The Phred score encoding system:

image [here](https://drive.google.com/file/d/1WNVk2h8AFKKcXyxSA2h0dyEAcRlVkKzL/view?usp=drive_link)

# Running MetaPhlAn

In [ ]:
# metaphlan database
metaphlan_db_dir = os.path.join(root_dir,"Metaphlan3_DB")
! ls {metaphlan_db_dir}

total 0
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_latest
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_previous
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.1.bt2
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.2.bt2
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.3.bt2
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.4.bt2
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.fna.bz2
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.md5
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.pkl
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.rev.1.bt2
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.rev.2.bt2
-rw-rw-rw- 1 root root 0 Oct  2 13:17 mpa_v31_CHOCOPhlAn_201901.tar
-rw-rw-rw- 1 root root 0 Oct  2 13:17 README.txt


## Command format:

In [13]:
! metaphlan --help

usage: metaphlan --input_type {fastq,fasta,bowtie2out,sam} [--force]
                 [--bowtie2db METAPHLAN_BOWTIE2_DB] [-x INDEX]
                 [--bt2_ps BowTie2 presets] [--bowtie2_exe BOWTIE2_EXE]
                 [--bowtie2_build BOWTIE2_BUILD] [--bowtie2out FILE_NAME]
                 [--min_mapq_val MIN_MAPQ_VAL] [--no_map] [--tmp_dir]
                 [--tax_lev TAXONOMIC_LEVEL] [--min_cu_len]
                 [--min_alignment_len] [--add_viruses] [--ignore_eukaryotes]
                 [--ignore_bacteria] [--ignore_archaea] [--stat_q]
                 [--perc_nonzero] [--ignore_markers IGNORE_MARKERS]
                 [--avoid_disqm] [--stat] [-t ANALYSIS TYPE]
                 [--nreads NUMBER_OF_READS] [--pres_th PRESENCE_THRESHOLD]
                 [--clade] [--min_ab] [-o output file] [--sample_id_key name]
                 [--use_group_representative] [--sample_id value]
                 [-s sam_output_file] [--legacy-output] [--CAMI_format_output]
                 [--u

Relavent parameters
*   rel_ab_w_read_stats: profiling relative abundances and estimating the number of reads coming from each clade.
*   nproc: The number of CPUs to use for parallelizing the mapping step [default 4]




In [14]:
# check input fastq files
! ls -lh {mgx_reads_dir}/{sample_id}_R*.fastq.gz
# create the output folder
! mkdir -p {metaphlan_profile_dir}

-rw-rw-rw- 1 root root 389M Sep 30 06:54 /biodata/resources/day1_lab2/mgx_reads/PSMB4MBK_R1.fastq.gz
-rw-rw-rw- 1 root root 391M Sep 30 06:54 /biodata/resources/day1_lab2/mgx_reads/PSMB4MBK_R2.fastq.gz


In [15]:
!echo {metaphlan_db_dir}

/biodata/resources/day1_lab2/Metaphlan3_DB


This is the command for running metaphlan. As you would usually execute this on a server with more compute power, we provide the pre-computed output below.

In [ ]:
#DO NOT EXECUTE
! zcat {mgx_reads_dir}/{sample_id}_R*.fastq.gz | metaphlan --bowtie2db {metaphlan_db_dir} -x mpa_v31_CHOCOPhlAn_201901 -t rel_ab_w_read_stats --nproc 8 --input_type fastq > {metaphlan_profile_dir}/{sample_id}.profile.txt;

## Check the output:

In [16]:
! ls -lh {metaphlan_profile_dir}/{sample_id}.profile.txt;
! head {metaphlan_profile_dir}/{sample_id}.profile.txt

-rw-rw-rw- 1 root root 23K Oct  2 03:40 /biodata/resources/day1_lab2/reference_based_metagenomic_analysis/profile/PSMB4MBK.profile.txt
#mpa_v31_CHOCOPhlAn_201901
#/opt/conda/bin/metaphlan --bowtie2db /biodata/resources/day1_lab2/reference_based_metagenomic_analysis/Metaphlan3_DB -x mpa_v31_CHOCOPhlAn_201901 -t rel_ab_w_read_stats --nproc 8 --input_type fastq
#SampleID	Metaphlan_Analysis
#estimated_reads_mapped_to_known_clades:5209240
#clade_name	clade_taxid	relative_abundance	coverage	estimated_number_of_reads_from_the_clade
k__Bacteria	2	100.0	1.43574	5209240
k__Bacteria|p__Firmicutes	2|1239	65.85502	0.94551	3005454
k__Bacteria|p__Bacteroidetes	2|976	32.28664	0.46355	2143004
k__Bacteria|p__Actinobacteria	2|201174	1.81638	0.02608	59545
k__Bacteria|p__Proteobacteria	2|1224	0.04196	0.0006	1237


In [17]:
# An example of bacterial lineage
! grep s__ -m1 {metaphlan_profile_dir}/{sample_id}.profile.txt #| cut -f1 | sed 's/|/\n/g'

k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii	2|1239|186801|186802|541000|216851|853	26.41677	0.37928	1135333


## Merging tables from multiple samples

In [18]:
! ls -lh {metaphlan_profile_dir}/*.profile3.txt

-rw-rw-rw- 1 root root 8.8K Oct  2 03:40 /biodata/resources/day1_lab2/reference_based_metagenomic_analysis/profile/CSM5FZ3T_P.profile3.txt
-rw-rw-rw- 1 root root  27K Oct  2 03:40 /biodata/resources/day1_lab2/reference_based_metagenomic_analysis/profile/CSM5FZ3V_P.profile3.txt


In [19]:
! merge_metaphlan_tables.py {metaphlan_profile_dir}/*profile3.txt > {metaphlan_profile_dir}/merged_abundance_table.txt

In [20]:
# check the output
! head {metaphlan_profile_dir}/merged_abundance_table.txt


#mpa_v30_CHOCOPhlAn_201901
clade_name	NCBI_tax_id	CSM5FZ3V_P.profile3	CSM5FZ3T_P.profile3
k__Bacteria	2	100.0	100.0
k__Bacteria|p__Bacteroidetes	2|976	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia	2|976|200643	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales	2|976|200643|171549	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae	2|976|200643|171549|815	92.6905	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides	2|976|200643|171549|815|816	92.6905	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_caccae	2|976|200643|171549|815|816|47678	2.70692	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_dorei	2|976|200643|171549|815|816|357276	0.84337	0.28864


In [21]:
# check species level abundance
#--line-buffered option may reduce buffering, so that you would not see the error "grep: write error: Broken pipe"
! grep --line-buffered "s__" {metaphlan_profile_dir}/merged_abundance_table.txt | head

k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_caccae	2|976|200643|171549|815|816|47678	2.70692	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_dorei	2|976|200643|171549|815|816|357276	0.84337	0.28864
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_fragilis	2|976|200643|171549|815|816|817	9.18445	11.08387
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_ovatus	2|976|200643|171549|815|816|28116	0.0776	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_salyersiae	2|976|200643|171549|815|816|291644	0.01008	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_thetaiotaomicron	2|976|200643|171549|815|816|818	11.28384	13.55901
k__Bact

In [22]:
# check phylym level abundance
! grep "p__" {metaphlan_profile_dir}/merged_abundance_table.txt | grep -v "c__"


k__Bacteria|p__Bacteroidetes	2|976	92.7779	95.6901
k__Bacteria|p__Firmicutes	2|1239	6.4211	3.8043
k__Bacteria|p__Proteobacteria	2|1224	0.03646	0.5056
k__Bacteria|p__Verrucomicrobia	2|74201	0.76454	0


In [23]:
# check order Bacteroidales at family level abundance
! grep "o__Bacteroidales" {metaphlan_profile_dir}/merged_abundance_table.txt | grep -v "g__"

k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales	2|976|200643|171549	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae	2|976|200643|171549|815	92.6905	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Rikenellaceae	2|976|200643|171549|171550	0.07243	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Tannerellaceae	2|976|200643|171549|2005525	0.01497	0


## Task: how many species were identified?

Hint:


*   **grep** species-level records
*   use **wc** command to count the number of records



In [24]:
# solution
! grep "s__" {metaphlan_profile_dir}/merged_abundance_table.txt | wc -l

48
